# Day 7 Practice: Optimization & Ensembling (Grandmaster Level)

**Role:** Competitive Kaggle Grandmaster

Today, you go for the gold. You will build a complete optimization and ensembling pipeline: tuning models with Optuna, extracting OOF predictions, blending model outputs, and finally performing a post-processing threshold sweep to maximize your metric.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import sys
import os

# Setup validation
try:
    import ipytest
except ImportError:
    !pip install -q ipytest
    import ipytest

if not os.path.exists('validate_day7.py'):
    print('⚠️ validate_day7.py not found!')
else:
    import validate_day7

ipytest.autoconfig()


## 1. Dataset Generation
Generating a synthetic imbalanced dataset.


In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, weights=[0.9, 0.1], random_state=42)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(20)])
df['Target'] = y


## Challenge 1: Optuna Objective Function
**Task:** Write an objective function to tune 5 LightGBM params (`num_leaves`, `learning_rate`, `n_estimators`, `max_depth`, `subsample`) over 20 trials.


In [ ]:
# YOUR CODE HERE
# def objective(trial):
# ... return CV score


In [ ]:
%%ipytest -qq
try:
    validate_day7.test_task_1()
except NameError: pass


## Challenge 2: OOF Extraction
**Task:** Run optimized LGBM and XGBoost models using 5-Fold CV. Store OOF probabilities in `lgbm_oof` and `xgb_oof`.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day7.test_task_2()
except NameError: pass


## Challenge 3: Blending
**Task:** Compute a weighted blend: `blend_preds = w * lgbm_oof + (1-w) * xgb_oof`. Find the `w` (0.0 to 1.0) that maximizes the F1 score.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day7.test_task_3()
except NameError: pass


## Challenge 4: Post-Processing Threshold Search
**Task:** Sweep probability thresholds from 0.01 to 0.99. Find the `best_threshold` that yields the maximum F1-score.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day7.test_task_4()
except NameError: pass


<details>
<summary style='color: #d35400; font-weight: bold; cursor: pointer;'>🏆 Click for Grandmaster Optimized Solutions</summary>

```python
# 1. Optuna
def objective(trial):
    params = { 'num_leaves': trial.suggest_int('nl', 20, 100), 'learning_rate': trial.suggest_float('lr', 0.01, 0.3), ... }
    # ... CV ...
study = optuna.create_study(direction='maximize').optimize(objective, n_trials=20)
best_params = study.best_params

# 2. OOF
# ... CV loop extracting lgbm_oof and xgb_oof ...

# 3. Blending
scores = [f1_score(y, (w * lgbm_oof + (1-w) * xgb_oof) > 0.5) for w in np.linspace(0,1,10)]
blend_preds = 0.5 * lgbm_oof + 0.5 * xgb_oof

# 4. Threshold search
thresholds = np.linspace(0.01, 0.99, 100)
scores = [f1_score(y, blend_preds > t) for t in thresholds]
best_threshold = thresholds[np.argmax(scores)]
best_f1 = max(scores)
```
</details>
